# Day 7: Streamlit Stock-Cluster Dashboard

**Goal:** turn the Day-5 K-Means + PCA analysis into an interactive web app -- a slider for the number of clusters `k`, a dropdown for picking a ticker, a live scatter plot of the clusters, and the PCA biplot -- then learn how to deploy it for free.

**Connection to Day 5:** the data and the analysis are exactly the ones from `Day5_UL_KMeans_PCA.ipynb` (30 synthetic stocks, 252 trading days, three hidden groups: bonds, growth, tech). Day 7 does not change the analysis. It wraps it in a dashboard.


### Setup: install Streamlit

The next cell installs Streamlit in Colab. Run it once; it takes about 30 seconds. We will write `app.py` files and (optionally) preview them via a tunnel. `# [FULL]`

In [ ]:
!pip -q install streamlit==1.31.1
print('Streamlit installed. We will write app.py files and (optionally) preview them via a tunnel.')

## Step 1: Rebuild the Day-5 stock data (verbatim)

The next two cells are copied unchanged from `Day5_UL_KMeans_PCA.ipynb` (commit `0a7ad3a9`). They import the libraries and build the 252-day, 30-stock return matrix with its three hidden groups. The dashboard's `app.py` will rebuild this same data so it is self-contained.

In [ ]:
# source: Day5_UL_KMeans_PCA.ipynb @ 0a7ad3a9
# Cell 0: Imports and colours
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

NAVY  = '#1e3a5f'
AMBER = '#d97706'
SLATE = '#64748b'


In [ ]:
# source: Day5_UL_KMeans_PCA.ipynb @ 0a7ad3a9
# Cell 1: Generate synthetic stock returns (3 hidden clusters)
n_days = 252

# Cluster 0: 10 low-vol bond instruments (mean ~0.02% / day, vol ~0.4%)
bond_r   = np.random.normal(0.0002, 0.004, (n_days, 10))
# Cluster 1: 10 growth equities (mean ~0.08% / day, vol ~1.8%)
growth_r = np.random.normal(0.0008, 0.018, (n_days, 10))
# Cluster 2: 10 high-vol tech stocks (mean ~0.10% / day, vol ~3.5%)
tech_r   = np.random.normal(0.001,  0.035, (n_days, 10))

returns = np.hstack([bond_r, growth_r, tech_r])   # (252, 30)

tickers = (
    [f'BOND{i+1:02d}' for i in range(10)] +
    [f'GROW{i+1:02d}' for i in range(10)] +
    [f'TECH{i+1:02d}' for i in range(10)]
)

df = pd.DataFrame(returns, columns=tickers)
true_labels = np.array([0]*10 + [1]*10 + [2]*10)   # ground truth for reference

print(f'Shape: {df.shape}')   # (252, 30)
print()
print(df.std().groupby(true_labels).mean().rename({0: 'Bonds', 1: 'Growth', 2: 'Tech'})
      .rename('Avg daily std').round(4).to_frame())


In [ ]:
# [FULL] Quick recap of the Day-5 analysis (so we know what the dashboard will show)
scaler = StandardScaler()
X_norm = scaler.fit_transform(df)            # (252, 30): standardize each stock's returns

pca = PCA(random_state=42)
pca.fit(X_norm)
stock_pca = pca.components_[:2].T            # (30, 2): each stock's loading on PC1 and PC2

km = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = km.fit_predict(stock_pca)
print('PC1 explains', f'{pca.explained_variance_ratio_[0]:.1%}', 'of the variance.')
print('Cluster sizes with k=3:', np.bincount(labels))
print('The dashboard lets a non-coder change k and the ticker and see this update live.')

## Step 2: What is Streamlit? (read this before any code)

**Streamlit turns a Python script into a web app.** You write an ordinary `.py` file, top to bottom. Every `st.slider(...)` becomes a control the user can drag. Every `st.pyplot(fig)` becomes a chart on the page. Every `st.write(...)` prints text or a table. There is no HTML, no CSS, no JavaScript -- it is the same Python you already know, plus a handful of `st.` calls.

The mental model: it is PowerPoint for data apps. PowerPoint lets you make a presentation without knowing graphic design; Streamlit lets you make a web app without knowing front-end development. You focus on the data and the model; Streamlit draws the page.

One quirk to know up front (it explains the `@st.cache_data` lesson later): **when the user moves any widget, Streamlit re-runs your ENTIRE script from the top.** Drag the `k` slider from 3 to 4? The whole file runs again with `k = 4`. That is what makes it feel live. It is also why, if your script re-loads a big dataset or re-fits a slow model on every run, the app crawls -- and why we cache those steps.

### Instructor demo: the smallest possible Streamlit app

Here is a three-line app: a title, one slider, and a line of text that reacts to the slider. We write it to disk with the `%%writefile` magic (it does not run inside Colab -- Colab is not a browser -- but it shows the shape). Outside Colab you would run it with `streamlit run demo_app.py` and a browser tab would open at `localhost:8501`.

In [ ]:
%%writefile demo_app.py
# [FULL]
# INSTRUCTOR DEMO: the smallest Streamlit app -- title, one slider, reactive text.
import streamlit as st

st.title('My First Streamlit App')
n = st.slider('Pick a number', min_value=1, max_value=100, value=10)
st.write(f'You picked {n}. Its square is {n * n}.')
# To run this on your own machine:  streamlit run demo_app.py

In [ ]:
# [FULL] Show what we just wrote (so students can see the file without leaving Colab).
print(open('demo_app.py').read())
print('---')
print('You would run this with:  streamlit run demo_app.py')
print('A browser tab opens at http://localhost:8501 with a draggable slider.')

## CHECKPOINT 1: drag `k` from 3 to 6 -- what changes in the scatter plot?

Before we build the dashboard, predict (discuss with your neighbour): the scatter plot colours each stock by its K-Means cluster. The user drags the `k` slider from 3 up to 6. What happens on screen? Do the points move? Does anything else change?

Your prediction: ________

## Step 3: The wrong way first -- recomputing everything on every interaction

Remember: Streamlit re-runs the whole script when any widget moves. Here is an `app.py` that re-builds the 30-stock dataset AND re-fits K-Means from scratch every single time the user touches the slider or the dropdown. With our tiny synthetic data it is merely wasteful; with a real dataset (millions of rows, a model that takes seconds to fit) the app would freeze for a few seconds on every click. We write it here so you can see the smell -- then immediately fix it.

In [ ]:
%%writefile app_slow.py
# [MIN]
# WRONG WAY: no @st.cache_data -- the data is rebuilt and K-Means is re-fit on EVERY rerun,
# i.e. every time the user drags the slider. Slow and pointless.
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

st.title('Stock Clusters (SLOW version -- do not ship this)')

def build_data():
    np.random.seed(42)
    bond_r   = np.random.normal(0.0002, 0.004, (252, 10))
    growth_r = np.random.normal(0.0008, 0.018, (252, 10))
    tech_r   = np.random.normal(0.001,  0.035, (252, 10))
    returns = np.hstack([bond_r, growth_r, tech_r])
    tickers = ([f'BOND{i+1:02d}' for i in range(10)] +
               [f'GROW{i+1:02d}' for i in range(10)] +
               [f'TECH{i+1:02d}' for i in range(10)])
    return pd.DataFrame(returns, columns=tickers)

# NO CACHE: this whole block re-runs on every slider move.
df = build_data()
X_norm = StandardScaler().fit_transform(df)
stock_pca = PCA(random_state=42).fit(X_norm).components_[:2].T

k = st.slider('Number of clusters k', 2, 8, 3)
labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(stock_pca)  # re-fit every time!

fig, ax = plt.subplots()
ax.scatter(stock_pca[:, 0], stock_pca[:, 1], c=labels, cmap='viridis')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
st.pyplot(fig)
st.caption('Every drag of the slider re-built the data and re-fit K-Means. On real data this lags.')

### The fix: `@st.cache_data` so the expensive steps run once

Wrap the slow, deterministic steps -- building the data, standardizing, computing the PCA coordinates -- in a function decorated with `@st.cache_data`. The first time the app runs, Streamlit executes the function and stores the result. Every later rerun (every slider drag) returns the stored result instantly. K-Means with the chosen `k` is cheap enough to leave outside the cache, but you could cache it per-`k` too. (Side note: `@st.cache_resource` is the variant for things like loaded ML models or database connections that should not be copied.)

In [ ]:
%%writefile app_fast.py
# [MIN]
# FIX: @st.cache_data -- the data + PCA are computed ONCE, reused on every rerun.
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

st.title('Stock Clusters (cached version -- this one is snappy)')

@st.cache_data
def load_pca_coords():
    np.random.seed(42)
    bond_r   = np.random.normal(0.0002, 0.004, (252, 10))
    growth_r = np.random.normal(0.0008, 0.018, (252, 10))
    tech_r   = np.random.normal(0.001,  0.035, (252, 10))
    returns = np.hstack([bond_r, growth_r, tech_r])
    tickers = ([f'BOND{i+1:02d}' for i in range(10)] +
               [f'GROW{i+1:02d}' for i in range(10)] +
               [f'TECH{i+1:02d}' for i in range(10)])
    df = pd.DataFrame(returns, columns=tickers)
    X_norm = StandardScaler().fit_transform(df)
    stock_pca = PCA(random_state=42).fit(X_norm).components_[:2].T
    return tickers, stock_pca

tickers, stock_pca = load_pca_coords()   # runs once; instant on every later rerun

k = st.slider('Number of clusters k', 2, 8, 3)
labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(stock_pca)

fig, ax = plt.subplots()
ax.scatter(stock_pca[:, 0], stock_pca[:, 1], c=labels, cmap='viridis')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
st.pyplot(fig)
st.caption('The data + PCA were cached. Sliding k just re-colours -- no rebuild.')

## CHECKPOINT 2: why does the app feel sluggish without `@st.cache_data`?

Predict (discuss with your neighbour): in the SLOW version above, what exactly is happening each time the user nudges the slider that makes it lag? Why would `@st.cache_data` fix it?

Your prediction: ________

## Step 4: The complete dashboard -- `app.py`

Now the real thing. This `app.py` has everything the brief asks for:

- loads the 30-stock returns (the same verbatim Day-5 data), cached with `@st.cache_data`;
- `st.slider` for `k`, from 2 to 8;
- `st.selectbox` for picking a ticker;
- `st.pyplot` of a scatter plot of the K-Means clusters in PC1/PC2 space, with the chosen ticker highlighted;
- `st.pyplot` of the PCA biplot (stocks coloured by their true type: bond / growth / tech).

We write it to disk; the cell after explains how to preview it.

In [ ]:
%%writefile app.py
# [FULL] Day 7 Streamlit dashboard: K-Means clusters + PCA biplot on the Day-5 stock data.
# Run locally with:  streamlit run app.py     (opens http://localhost:8501)
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

NAVY  = '#1e3a5f'
AMBER = '#d97706'
SLATE = '#64748b'

st.set_page_config(page_title='Stock Cluster Dashboard', layout='wide')
st.title('Stock Cluster Dashboard')
st.caption('30 synthetic stocks, 252 trading days, three hidden groups. Day 5 analysis, now interactive.')

# ---- data + PCA: expensive, deterministic -> cache it (computed once) ----
@st.cache_data
def load_data():
    # source: Day5_UL_KMeans_PCA.ipynb @ 0a7ad3a9  (same generators)
    np.random.seed(42)
    bond_r   = np.random.normal(0.0002, 0.004, (252, 10))
    growth_r = np.random.normal(0.0008, 0.018, (252, 10))
    tech_r   = np.random.normal(0.001,  0.035, (252, 10))
    returns = np.hstack([bond_r, growth_r, tech_r])
    tickers = ([f'BOND{i+1:02d}' for i in range(10)] +
               [f'GROW{i+1:02d}' for i in range(10)] +
               [f'TECH{i+1:02d}' for i in range(10)])
    df = pd.DataFrame(returns, columns=tickers)
    true_labels = np.array([0]*10 + [1]*10 + [2]*10)
    X_norm = StandardScaler().fit_transform(df)
    pca = PCA(random_state=42).fit(X_norm)
    stock_pca = pca.components_[:2].T          # (30, 2): loadings on PC1, PC2
    return tickers, true_labels, stock_pca, pca.explained_variance_ratio_

tickers, true_labels, stock_pca, evr = load_data()

# ---- sidebar controls ----
st.sidebar.header('Controls')
k = st.sidebar.slider('Number of clusters k', min_value=2, max_value=8, value=3,
                      help='Day 5 elbow plot says 3 is the honest answer. Try sliding past it.')
ticker = st.sidebar.selectbox('Highlight a ticker', tickers, index=0)

# ---- K-Means with the chosen k (cheap; fine to recompute each rerun) ----
labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(stock_pca)
ticker_idx = tickers.index(ticker)

col1, col2 = st.columns(2)

# ---- left: K-Means cluster scatter, chosen ticker highlighted ----
with col1:
    st.subheader(f'K-Means clusters (k = {k})')
    fig1, ax1 = plt.subplots(figsize=(6, 5))
    sc = ax1.scatter(stock_pca[:, 0], stock_pca[:, 1], c=labels, cmap='viridis',
                     s=70, edgecolors='white', linewidths=0.5, zorder=3)
    ax1.scatter(stock_pca[ticker_idx, 0], stock_pca[ticker_idx, 1],
                s=320, facecolors='none', edgecolors=AMBER, linewidths=2.5, zorder=5,
                label=f'{ticker} (selected)')
    ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2')
    ax1.set_title(f'{ticker} is in cluster {labels[ticker_idx]}', color=NAVY)
    ax1.legend(loc='best', fontsize=8)
    st.pyplot(fig1)

# ---- right: PCA biplot, coloured by TRUE stock type ----
with col2:
    st.subheader('PCA biplot (true stock types)')
    type_colors = {0: NAVY, 1: AMBER, 2: SLATE}
    type_names  = {0: 'Bonds', 1: 'Growth', 2: 'Tech'}
    fig2, ax2 = plt.subplots(figsize=(6, 5))
    for t in range(3):
        mask = true_labels == t
        ax2.scatter(stock_pca[mask, 0], stock_pca[mask, 1],
                    color=type_colors[t], s=70, label=type_names[t], zorder=3)
    ax2.scatter(stock_pca[ticker_idx, 0], stock_pca[ticker_idx, 1],
                s=320, facecolors='none', edgecolors='black', linewidths=2.0, zorder=5)
    ax2.axhline(0, color=SLATE, lw=0.5, ls='--'); ax2.axvline(0, color=SLATE, lw=0.5, ls='--')
    ax2.set_xlabel(f'PC1 ({evr[0]:.0%} of variance)')
    ax2.set_ylabel(f'PC2 ({evr[1]:.0%} of variance)')
    ax2.set_title('Stocks in principal-component space', color=NAVY)
    ax2.legend(loc='best', fontsize=8)
    st.pyplot(fig2)

st.divider()
st.write(f'Selected **{ticker}** -> K-Means cluster **{labels[ticker_idx]}** (true type: **{type_names[true_labels[ticker_idx]]}**).')
st.caption('Drag k past 3 and watch the bond cluster get sliced into meaningless pieces. '
           'Day 5 elbow analysis is why k = 3 is the right call.')

In [ ]:
%%writefile requirements.txt
streamlit
scikit-learn
numpy
pandas
matplotlib

## Step 5: Running it and deploying it

### Previewing inside Colab (works, but flaky)

Colab has no public port, so to *see* the app you tunnel it out. The usual recipe:

```bash
!streamlit run app.py &>/dev/null &          # start the app in the background
!npx --yes localtunnel --port 8501           # or: cloudflared tunnel --url http://localhost:8501
```

This prints a temporary public URL. **Treat it as unreliable**: the tunnel can be blocked on school or corporate networks, the URL dies when the Colab runtime resets, and `localtunnel` sometimes shows an interstitial password page (the password is the runtime's public IP, which the tunnel prints). It is fine for a quick look in class; it is not how you ship the app.

### The canonical way to deploy: Hugging Face Spaces (free, no GitHub needed)

This is the path to a real, permanent URL that anyone can open -- and you never touch the command line:

1. Go to `huggingface.co/spaces` and click **Create new Space**.
2. Give it a name, pick **Streamlit** as the SDK, choose the free CPU hardware, set it Public.
3. In the Space's **Files** tab, click **Add file -> Upload files** and drag in `app.py` and `requirements.txt` (the two files this notebook just wrote). That is it -- no `git`, no public repo required.
4. The Space builds automatically (a minute or two) and your dashboard is live at `huggingface.co/spaces/<your-username>/<space-name>`. Share that link.

### If you happen to have a public GitHub repo

Then **Streamlit Community Cloud** (`share.streamlit.io`) is also an option: connect the repo, point it at `app.py`, click Deploy. It only works with public repos, which is why Hugging Face Spaces (browser drag-and-drop, no repo) is the default we teach.

## What you just built

- An `app.py` that loads the Day-5 stock data (cached with `@st.cache_data`), with a `k` slider (2 to 8), a ticker dropdown, a live K-Means cluster scatter plot, and the PCA biplot.
- A `requirements.txt` so the app runs anywhere.
- The knowledge of why caching matters (Streamlit re-runs the whole script on every interaction) and how to deploy for free to Hugging Face Spaces by dragging two files into a browser.

The K-Means + PCA analysis is unchanged from Day 5. We just gave it sliders and a URL.